# Exploração da base Olist

Só exploração visual. Os números oficiais vêm de `scripts/eda_resumo.py` (arquivo `docs/eda-resumo.md`).

Antes de rodar: `python scripts/dbt_build.py`.

In [ ]:
from pathlib import Path

import duckdb
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
con = duckdb.connect(str(ROOT / "data" / "processed" / "copiloto.duckdb"), read_only=True)

## Volume de pedidos por mês

In [ ]:
monthly = con.sql(
    "select date_trunc('month', purchased_at) as mes, count(*) as pedidos "
    "from staging.stg_orders group by mes order by mes"
).df()
monthly.plot(x="mes", y="pedidos", legend=False, title="Pedidos por mês")
plt.show()

## Distribuição das notas

In [ ]:
scores = con.sql(
    "select review_score, count(*) as n from intermediate.int_order_reviews group by 1 order by 1"
).df()
scores.plot.bar(x="review_score", y="n", legend=False, title="Pedidos por nota")
plt.show()

## Taxa de nota negativa por dias de atraso

Correlação, não efeito causal: a Fase 3 controla os confundidores.

In [ ]:
by_delay = con.sql(
    "select round(greatest(least(delay_days, 30), -30)) as atraso_dias, "
    "avg(is_negative::int) as taxa_negativa, count(*) as n "
    "from marts.fct_orders_risk group by 1 having count(*) >= 30 order by 1"
).df()
by_delay.plot(
    x="atraso_dias", y="taxa_negativa", legend=False, title="Taxa negativa x atraso (dias)"
)
plt.axvline(0, color="gray", linestyle="--")
plt.show()

## Tamanho dos comentários das reclamações

In [ ]:
lengths = con.sql("select comment_length from marts.fct_complaints").df()
lengths["comment_length"].clip(upper=500).plot.hist(bins=50, title="Caracteres por reclamação")
plt.show()

In [ ]:
con.close()